# Prerequisites

In [0]:
from pyspark.sql.functions import *
# customers_df = spark.read.table("ecommerce.silver.customers")
# orders_df = spark.read.table("ecommerce.silver.orders")
order_items_df = spark.read.table("ecommerce.silver.order_items")
products_df = spark.read.table("ecommerce.silver.products")
# payments_df = spark.read.table("ecommerce.silver.payments")
inventory_df = spark.read.table("ecommerce.silver.inventory")
# clickstream_df = spark.read.table("ecommerce.silver.clickstream")

# Calculations

## Combine: Sales, Inventory, Products

In [0]:
sales_qty_df = order_items_df.groupBy("product_id").agg(sum("quantity").alias("sales_quantity"))
sales_qty_df.limit(5).display()


In [0]:
prod_sales = products_df.join(sales_qty_df, "product_id", "left").join(
    inventory_df, "product_id", "left"
)
prod_sales.limit(5).display()

In [0]:
prod_sales.select("stock_status").distinct().display()

## available_inventory, reserved_inventory, inventory_status

In [0]:
inventory_sales = prod_sales.select(
    "product_id",
    "product_name",
    "sales_quantity",
    col("available_to_sell").alias("available_inventory"),
    col("reserved_quantity").alias("reserved_inventory")
).withColumn(
    "inventory_status",
    when(col("available_inventory") == 0, "OUT_OF_STOCK")
    .when(col("available_inventory") <= 10, "LOW_STOCK")
    .otherwise("HEALTHY_STOCK"),
)

inventory_sales.limit(5).display()

# Create gold.inventory_sales Delta table

In [0]:
inventory_sales.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    "ecommerce.gold.inventory_sales"
)
spark.read.table("ecommerce.gold.inventory_sales").limit(5).display()

# Business Questions

In [0]:
inventory_sales = spark.read.table("ecommerce.gold.inventory_sales")

## Which high-selling products are out of stock?


In [0]:
avg_sales = inventory_sales.select(avg("sales_quantity").alias("avg_sales")).collect()[
    0
]["avg_sales"]

display(avg_sales)

inventory_sales.filter(
    (col("sales_quantity") > avg_sales) & (col("inventory_status") == "OUT_OF_STOCK")
).sort(desc(col("sales_quantity"))).limit(5).display()

## Which products need replenishment?

In [0]:
inventory_sales.filter(col("inventory_status") != "HEALTHY_STOCK").sort(
    desc(col("sales_quantity"))
).select("product_name", "available_inventory").limit(5).display()

## Which products have excess inventory?

In [0]:
inventory_sales.withColumn(
    "inventory_ratio",
    round(try_divide(col("available_inventory"), col("sales_quantity")), 2),
).sort(desc(col("inventory_ratio"))).select(
    "product_name", "inventory_ratio", "available_inventory", "sales_quantity"
).limit(5).display()